# Climate Change Dashboard — Google Colab Walkthrough

**Lumexa Data Scientist Path — Course 14, Data Visualisation**

Build a real, interactive web dashboard exploring how CO2 emissions have changed over
time across major countries, using [Our World in Data](https://github.com/owid/co2-data)'s
public CO2 & Greenhouse Gas Emissions dataset. This notebook is the capstone application
of Lessons 6-8 (Interactive charts with Plotly, Building a Dash dashboard, Final dashboard).

This notebook is fully self-contained and works with **Runtime → Run all** — no local
files to upload, no API keys, and no accounts. The real dataset is downloaded directly
from its public source at runtime.

**What you'll do:**
1. Download the real, full OWID CO2 dataset and trim it down to a classroom-sized subset
2. Write plain Python functions that compute KPIs and build the dashboard's Plotly charts
3. Test those functions directly on several sample inputs, so you can see real numbers
   and real charts render right in the notebook — no live app needed to check your work
4. Assemble those functions into a full interactive [Dash](https://dash.plotly.com/) app
   with dropdown-driven callbacks and linked interactivity
5. Launch the app inline, right inside this notebook (Dash's built-in Jupyter/Colab
   support handles this automatically — no `ngrok` or extra setup)

**What you'll learn:**
- Loading, cleaning, and filtering a real-world CSV dataset with pandas.
- Building a multi-feature, single-page Dash app with dropdown-driven callbacks.
- Designing linked interactivity: one dropdown updates multiple charts and KPI cards at once.
- Choosing chart types (line for trends, stacked area for composition, multi-line for
  comparison) appropriate to each question.
- Applying honest, accessible color choices.


## Step 1: Install packages

Google Colab already has `pandas` and `plotly` preinstalled. `dash` is usually not
preinstalled, so we install a current version (>= 2.11), which has built-in Jupyter/Colab
support for running interactive apps directly inside a notebook cell.

In [1]:
import importlib.util

# Only install what's actually missing in this environment.
missing = [pkg for pkg in ["dash", "plotly"] if importlib.util.find_spec(pkg) is None]
if missing:
    %pip install -q {" ".join(missing)}
else:
    print("dash and plotly already available, skipping install.")


dash and plotly already available, skipping install.


In [2]:
import dash
import plotly
import pandas as pd

print(f"dash version:   {dash.__version__}")
print(f"plotly version: {plotly.__version__}")
print(f"pandas version: {pd.__version__}")


dash version:   4.4.1
plotly version: 7.1.0
pandas version: 3.0.5


## Step 2: Download and trim the real OWID CO2 dataset

**Source:** Our World in Data, CO2 and Greenhouse Gas Emissions dataset
**URL:** `https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv`
**License:** Our World in Data publishes this dataset under a Creative Commons BY
license; it aggregates data from the Global Carbon Project and other sources.

The full upstream file contains **50,411 rows** across 254 countries/regions and years
1750-2024 (79 columns) — far more than a classroom dashboard needs. We download it fresh
every time this notebook runs (so it always reflects the latest published data) and then
filter it down to a small, manageable subset, exactly like a real analyst would when
scoping a dashboard: **no numbers are invented, only rows/columns are removed.**

**The trim:**
- **8 real countries/regions:** World, United States, China, India, Germany, Brazil,
  Australia, United Kingdom
- **Years 1950-2024**
- **18 relevant columns:** identifiers (`country`, `year`, `iso_code`), `population`,
  `gdp`, core emissions metrics (`co2`, `co2_per_capita`, `co2_growth_prct`,
  `cumulative_co2`), emissions by fossil-fuel source (`coal_co2`, `oil_co2`, `gas_co2`),
  other greenhouse gases (`methane`, `nitrous_oxide`), `temperature_change_from_co2`,
  `share_global_co2`, and energy metrics (`energy_per_capita`,
  `primary_energy_consumption`)

This produces **600 rows x 18 columns** (75 years x 8 countries/regions) — every value
kept is a real, unmodified measurement from the original OWID file.

In [3]:
OWID_CO2_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

# The 8 countries/regions and 18 columns kept for this classroom-sized dashboard.
KEPT_COUNTRIES = [
    "World", "United States", "China", "India",
    "Germany", "Brazil", "Australia", "United Kingdom",
]
KEPT_COLUMNS = [
    "country", "year", "iso_code", "population", "gdp",
    "co2", "co2_per_capita", "co2_growth_prct", "cumulative_co2",
    "coal_co2", "oil_co2", "gas_co2",
    "methane", "nitrous_oxide",
    "temperature_change_from_co2", "share_global_co2",
    "energy_per_capita", "primary_energy_consumption",
]
MIN_YEAR = 1950

print("Downloading the full OWID CO2 dataset (this may take a few seconds)...")
full_df = pd.read_csv(OWID_CO2_URL)
print(f"Full dataset: {full_df.shape[0]:,} rows x {full_df.shape[1]} columns")

df = (
    full_df[full_df["country"].isin(KEPT_COUNTRIES) & (full_df["year"] >= MIN_YEAR)]
    [KEPT_COLUMNS]
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

# The dashboard's primary metrics (co2, co2_per_capita) must be complete for every kept row.
df = df.dropna(subset=["co2", "co2_per_capita"])

print(f"Trimmed dataset:  {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Countries/regions kept: {sorted(df['country'].unique().tolist())}")
print(f"Year range: {int(df['year'].min())}-{int(df['year'].max())}")
df.head()


Full dataset: 50,411 rows x 79 columns
Trimmed dataset:  600 rows x 18 columns
Countries/regions kept: ['Australia', 'Brazil', 'China', 'Germany', 'India', 'United Kingdom', 'United States', 'World']
Year range: 1950-2024


,country,year,iso_code,population,gdp,co2,co2_per_capita,co2_growth_prct,cumulative_co2,coal_co2,oil_co2,gas_co2,methane,nitrous_oxide,temperature_change_from_co2,share_global_co2,energy_per_capita,primary_energy_consumption
0,Australia,1950,AUS,8176751.0,9.767859e+10,55.636,6.804,15.752,1531.930,43.968,11.035,NaN,80.725,38.059,0.004,0.938,NaN,NaN
1,Australia,1951,AUS,8418616.0,1.018378e+11,61.051,7.252,9.734,1592.981,46.026,14.414,NaN,83.949,38.792,0.004,0.957,NaN,NaN
2,Australia,1952,AUS,8630707.0,1.027649e+11,62.406,7.231,2.219,1655.388,46.987,14.747,NaN,86.567,39.580,0.004,0.965,NaN,NaN
3,Australia,1953,AUS,8816085.0,1.059673e+11,63.003,7.146,0.956,1718.391,48.199,14.015,NaN,89.038,40.418,0.004,0.947,NaN,NaN
4,Australia,1954,AUS,8999384.0,1.125660e+11,67.921,7.547,7.806,1786.311,51.113,15.863,NaN,92.297,41.299,0.004,1.000,NaN,NaN


In [4]:
# A quick, honest look at data completeness in the trimmed set (mirrors real-world
# exploratory data analysis -- some columns have gaps in the original OWID data too).
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values in the trimmed dataset:")
print(missing if len(missing) else "(none)")
print()
print("Note: 'iso_code' is null for 'World' rows -- expected, since it's an aggregate,")
print("not a real country with an ISO code.")


Columns with missing values in the trimmed dataset:
primary_energy_consumption    120
energy_per_capita             120
iso_code                       75
gdp                            74
gas_co2                        22
dtype: int64

Note: 'iso_code' is null for 'World' rows -- expected, since it's an aggregate,
not a real country with an ISO code.


## Step 3: Write the dashboard's analysis functions

Rather than putting all the logic inline inside Dash's `@app.callback` decorators, we
write plain Python functions first. This has two advantages:
- It keeps the Dash callback functions themselves thin and readable.
- It lets us **test the exact same logic the live app will use**, directly, without
  needing a running browser -- which is exactly what we do in Step 4 below.

Four functions do all the work:
- `compute_country_kpis(df, country)` -> a plain dict of the 4 KPI numbers
- `make_trend_chart(df, country)` -> a Plotly line chart of total CO2 over time
- `make_source_chart(df, country)` -> a Plotly stacked-area chart by fossil fuel source
- `make_comparison_charts(df, countries)` -> two Plotly multi-line charts (total and
  per-capita CO2) comparing several countries at once

In [5]:
import plotly.express as px

def compute_country_kpis(df, country):
    """Return a plain dict of the 4 headline KPI numbers for one country/region."""
    sub = df[df["country"] == country].sort_values("year")
    latest = sub.iloc[-1]
    earliest = sub.iloc[0]
    pct_change = (
        (latest["co2"] - earliest["co2"]) / earliest["co2"] * 100
        if earliest["co2"] else float("nan")
    )
    share = latest.get("share_global_co2", float("nan"))
    return {
        "country": country,
        "latest_year": int(latest["year"]),
        "earliest_year": int(earliest["year"]),
        "latest_total_co2_mt": round(float(latest["co2"]), 1),
        "co2_per_capita_t": round(float(latest["co2_per_capita"]), 2),
        "pct_change_since_earliest": round(float(pct_change), 1),
        "share_global_co2_pct": round(float(share), 1) if pd.notna(share) else None,
    }


def make_trend_chart(df, country):
    """Line chart: total CO2 emissions over time for one country/region."""
    sub = df[df["country"] == country].sort_values("year")
    earliest_year, latest_year = int(sub["year"].min()), int(sub["year"].max())
    fig = px.line(
        sub, x="year", y="co2", markers=False,
        title=f"{country}: Total CO2 Emissions, {earliest_year}-{latest_year}",
        labels={"year": "Year", "co2": "CO2 emissions (million tonnes)"},
        template="plotly_white",
    )
    fig.update_traces(line_color="#2a9d8f", line_width=3)
    return fig


def make_source_chart(df, country):
    """Stacked-area chart: CO2 emissions broken down by fossil fuel source."""
    sub = df[df["country"] == country].sort_values("year")
    source_cols = [c for c in ["coal_co2", "oil_co2", "gas_co2"] if c in sub.columns]
    source_df = sub[["year"] + source_cols].melt(
        id_vars="year", var_name="source", value_name="emissions"
    )
    source_df["source"] = source_df["source"].str.replace("_co2", "", regex=False).str.title()
    fig = px.area(
        source_df, x="year", y="emissions", color="source",
        title=f"{country}: CO2 Emissions by Fossil Fuel Source",
        labels={"year": "Year", "emissions": "CO2 emissions (million tonnes)", "source": "Source"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    return fig


def make_comparison_charts(df, countries):
    """Two multi-line charts comparing several countries: total CO2 and CO2 per capita."""
    if not countries:
        countries = ["World"]
    comp_df = df[df["country"].isin(countries)].sort_values(["country", "year"])

    comparison_fig = px.line(
        comp_df, x="year", y="co2", color="country",
        title="Total CO2 Emissions Over Time: Country Comparison",
        labels={"year": "Year", "co2": "CO2 emissions (million tonnes)", "country": "Country"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set1,
    )
    per_capita_fig = px.line(
        comp_df, x="year", y="co2_per_capita", color="country",
        title="CO2 Emissions Per Capita: Country Comparison",
        labels={"year": "Year", "co2_per_capita": "CO2 per capita (tonnes)", "country": "Country"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set1,
    )
    return comparison_fig, per_capita_fig


print("Analysis functions defined.")


Analysis functions defined.


## Step 4: Test the functions directly on sample inputs

Before wiring these functions into a live, clickable Dash app, let's call them directly
with a few concrete country selections and look at the real results -- exact KPI numbers
and real Plotly charts, rendered right here in the notebook. This is the most reliable
way to check the dashboard's logic is correct, since it doesn't depend on clicking
anything in a browser.

In [6]:
from IPython.display import display

for sample_country in ["World", "China", "United States"]:
    kpis = compute_country_kpis(df, sample_country)
    print(f"--- {sample_country} ---")
    for key, value in kpis.items():
        print(f"  {key}: {value}")
    print()


--- World ---
  country: World
  latest_year: 2024
  earliest_year: 1950
  latest_total_co2_mt: 38598.6
  co2_per_capita_t: 4.73
  pct_change_since_earliest: 550.9
  share_global_co2_pct: 100.0

--- China ---
  country: China
  latest_year: 2024
  earliest_year: 1950
  latest_total_co2_mt: 12289.0
  co2_per_capita_t: 8.66
  pct_change_since_earliest: 15488.3
  share_global_co2_pct: 31.8

--- United States ---
  country: United States
  latest_year: 2024
  earliest_year: 1950
  latest_total_co2_mt: 4904.1
  co2_per_capita_t: 14.2
  pct_change_since_earliest: 93.0
  share_global_co2_pct: 12.7



In [7]:
# Charts for "World"
display(make_trend_chart(df, "World"))
display(make_source_chart(df, "World"))


In [8]:
# Charts for "China" -- a different country, to confirm the functions really do
# respond to their input rather than always showing the same thing.
display(make_trend_chart(df, "China"))
display(make_source_chart(df, "China"))


In [9]:
# Charts for "United States"
display(make_trend_chart(df, "United States"))
display(make_source_chart(df, "United States"))


In [10]:
# Comparison charts, sample 1: the dashboard's default comparison set.
comparison_fig, per_capita_fig = make_comparison_charts(
    df, ["United States", "China", "India", "Germany"]
)
display(comparison_fig)
display(per_capita_fig)


In [11]:
# Comparison charts, sample 2: a different set of countries, to confirm the
# comparison functions genuinely respond to the selection.
comparison_fig_2, per_capita_fig_2 = make_comparison_charts(
    df, ["Brazil", "Australia", "United Kingdom"]
)
display(comparison_fig_2)
display(per_capita_fig_2)


## Step 5: Assemble the full interactive Dash app

Now we wire the same functions above into a real Dash app: a country dropdown drives 4
KPI cards plus the trend and source charts, and an independent multi-select dropdown
drives the two comparison charts. Because the callbacks below just call the functions we
already tested in Step 4, we know the logic itself is correct -- the only new thing here
is the interactive wiring.

In [12]:
from dash import Dash, dcc, html, Input, Output

countries = sorted(df["country"].unique().tolist())
default_country = "World" if "World" in countries else countries[0]
comparison_countries = [c for c in countries if c != "World"]
latest_year = int(df["year"].max())

app = Dash(__name__)
app.title = "Lumexa | Climate Change Dashboard"
server = app.server  # exposes the underlying Flask server for verification/testing


def kpi_card(label, value):
    return html.Div(
        style={"border": "1px solid #ddd", "borderRadius": "8px", "padding": "14px 22px",
               "textAlign": "center", "minWidth": "150px"},
        children=[
            html.Div(label, style={"fontSize": "12px", "color": "#888"}),
            html.Div(value, style={"fontSize": "22px", "fontWeight": "bold"}),
        ],
    )


app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "maxWidth": "1050px", "margin": "0 auto", "padding": "24px"},
    children=[
        html.H1("Climate Change Dashboard", style={"textAlign": "center"}),
        html.P(
            "Explore real CO2 emissions data from Our World in Data (1950-2024) for the "
            "World and 7 major countries. Select a country below to update every chart.",
            style={"textAlign": "center", "color": "#555"},
        ),
        html.Div(
            style={"textAlign": "center", "margin": "16px 0"},
            children=[
                html.Label("Country / Region: ", style={"fontWeight": "bold"}),
                dcc.Dropdown(
                    id="country-dropdown",
                    options=[{"label": c, "value": c} for c in countries],
                    value=default_country,
                    clearable=False,
                    style={"width": "320px", "margin": "8px auto"},
                ),
            ],
        ),

        html.Div(id="kpi-row", style={"display": "flex", "justifyContent": "center",
                                       "gap": "24px", "flexWrap": "wrap", "margin": "20px 0"}),

        dcc.Graph(id="co2-trend-chart"),
        dcc.Graph(id="co2-source-chart"),

        html.H3("Country Comparison", style={"textAlign": "center", "marginTop": "36px"}),
        html.Div(
            style={"textAlign": "center", "margin": "10px 0 20px"},
            children=[
                html.Label("Compare countries: ", style={"fontWeight": "bold"}),
                dcc.Dropdown(
                    id="compare-dropdown",
                    options=[{"label": c, "value": c} for c in comparison_countries],
                    value=["United States", "China", "India", "Germany"],
                    multi=True,
                    style={"width": "600px", "margin": "8px auto"},
                ),
            ],
        ),
        dcc.Graph(id="comparison-chart"),
        dcc.Graph(id="per-capita-chart"),

        html.P(
            f"Data source: Our World in Data CO2 & Greenhouse Gas Emissions dataset "
            f"(github.com/owid/co2-data), trimmed to 8 countries + World, 1950-{latest_year}. "
            f"Values are real, unmodified measurements for the rows kept.",
            style={"textAlign": "center", "color": "#999", "fontSize": "12px", "marginTop": "30px"},
        ),
    ],
)


@app.callback(
    Output("kpi-row", "children"),
    Output("co2-trend-chart", "figure"),
    Output("co2-source-chart", "figure"),
    Input("country-dropdown", "value"),
)
def update_country_view(selected_country):
    kpis = compute_country_kpis(df, selected_country)
    cards = [
        kpi_card("Latest Total CO2 (Mt)", f"{kpis['latest_total_co2_mt']:.1f}"),
        kpi_card("CO2 per Capita (t)", f"{kpis['co2_per_capita_t']:.2f}"),
        kpi_card(f"Change since {kpis['earliest_year']}", f"{kpis['pct_change_since_earliest']:+.0f}%"),
        kpi_card(
            "Share of Global CO2",
            f"{kpis['share_global_co2_pct']:.1f}%" if kpis["share_global_co2_pct"] is not None else "N/A",
        ),
    ]
    trend_fig = make_trend_chart(df, selected_country)
    source_fig = make_source_chart(df, selected_country)
    return cards, trend_fig, source_fig


@app.callback(
    Output("comparison-chart", "figure"),
    Output("per-capita-chart", "figure"),
    Input("compare-dropdown", "value"),
)
def update_comparison(selected_countries):
    return make_comparison_charts(df, selected_countries)


print("Dash app assembled:", app.title)
print("Callbacks registered:", [f.__name__ for f in [update_country_view, update_comparison]])


Dash app assembled: Lumexa | Climate Change Dashboard
Callbacks registered: ['update_country_view', 'update_comparison']


## Step 6: Launch the interactive dashboard

Modern `dash` (>= 2.11) can detect when it's running inside a notebook -- including
Google Colab -- and serve the app right there, using Colab's own port-forwarding, with
no `ngrok` or extra setup required. We try `jupyter_mode="inline"` first (the app appears
directly below this cell); if that's ever unavailable in a particular environment, we
fall back to `jupyter_mode="external"` (the app opens in a separate browser tab/URL
using the same automatic port-forwarding).

**In Google Colab:** just run this cell -- the dashboard will appear inline below it.
Try switching the country dropdown and the compare dropdown to see every chart and KPI
update live.

In [13]:
try:
    app.run(jupyter_mode="inline", port=8050, jupyter_height=850)
    print("Dash app started in 'inline' Jupyter mode.")
except Exception as inline_error:
    print(f"'inline' mode unavailable ({inline_error}); falling back to 'external' mode.")
    app.run(jupyter_mode="external", port=8050)
    print("Dash app started in 'external' Jupyter mode -- open the printed URL above.")


Dash app started in 'inline' Jupyter mode.


## Summary

This notebook builds a complete, real, interactive Dash dashboard on top of Our World in
Data's public CO2 emissions dataset:

1. **Real data, fetched live:** the full 50,000+ row OWID file is downloaded fresh every
   run and trimmed to 8 countries/regions and 1950-2024 -- 600 rows x 18 columns of real,
   unmodified measurements.
2. **Reusable analysis functions:** `compute_country_kpis`, `make_trend_chart`,
   `make_source_chart`, and `make_comparison_charts` compute every KPI and chart in the
   dashboard, and were tested directly on multiple sample countries before being wired
   into the app.
3. **A real interactive Dash app:** a country dropdown drives 4 KPI cards, a CO2 trend
   line, and a stacked-area chart by fossil fuel source; an independent multi-select
   dropdown drives two country-comparison line charts (total and per-capita CO2) --
   all linked, all updating instantly, all computed live.

**Try it yourself:**
- In the running dashboard above, switch the country dropdown to a few different
  countries and watch every chart and KPI update.
- Add a few more countries to `KEPT_COUNTRIES` in Step 2 and re-run the notebook
  (`Runtime -> Run all`) to expand the dashboard with more real data.
- Add a `dcc.RangeSlider` to filter the year range shown in each chart.
- Add a per-capita vs. total CO2 scatter chart, colored by country, to explore the
  relationship between population size and emissions.
